# HOW TO GLIMT

In [1]:
import load_secrets, os
load_secrets.load_secrets()

## JSX Requests

The JSX requests using curl have this format:

In [2]:
import os, json, requests

def jsx_request(jsxRequest):
    url = "https://glimt.nu/glimt-jsx/jsx.json?lang=en"
    
    headers = {
        "Content-Type": "application/json;charset=utf-8"
    }
    
    cookies = {
        "lumAuth": os.getenv("GLIMT_API_KEY")
    }
       
    response = requests.post(url, headers=headers, cookies=cookies, data=jsxRequest)
    
    print(response.status_code)

    return json.loads(response.text)

**Response text:** the text returned by a JSX request is itself always wrapped inside a JSON array. Therefore, below, when we say that a request returns value X, it really means that it returns [ X ] . 

If the response  text does not start by '[', i.e. is not a JSON Array, it indicates an error which is described more or less opaquely in the reply.

## Dates expressed as offsets from Unix day 0

In [99]:
from datetime import datetime, timedelta

def days_from_0_to_datetime(day):
    seconds_per_day =  86400
    days_in_s = (day-1) * seconds_per_day
    dt = datetime.fromtimestamp(days_in_s)-timedelta(hours=19)
    idt = int(str(dt)[0:10].replace('-',''))
    return idt

if __name__=="__main__":
    # should map 20104 to Jan. 14, 2025 and 20454 to Dec. 30, 2025. 
    print([(x, days_from_0_to_datetime(x)) for x in [20104, 20454]])

[(20104, 20250114), (20454, 20251230)]


## Query active IFPs

To request the list of active IFPs, replace jsxRequest by:

In [3]:
def list_active_ifps():
    L = jsx_request("""[["ifps", "queryIFPs", {query: {states: ["active"]}, fmt: {}}]]""")
    return L[0]

It will return a JSON array of all active iFPs and their detailed properties, including:
* **symbol**
* **title**
* **details**: Information beyond the title of the IFP, such as what sources may be used to resolve the IFP, and/or some background information that might be useful to forecasters, &c.
* **bins**: An array of the proposed resolution outcomes

In [102]:
ifps = list_active_ifps()

200


In [104]:
ifps_back = ifps.copy()

In [103]:
len(ifps)

19

## Convert dates in IFPs

In [105]:
for ifp in ifps:
    for key in ifp['dates']:
        value = ifp['dates'][key]
        ifp['dates'][key] = days_from_0_to_datetime(value)

In [106]:
ifps[10]['dates'][key]

20251230

## Save IFPs to disk

In [107]:
os.makedirs('glimt/ifp', exist_ok=True)

In [108]:
from tqdm import tqdm
import json

In [314]:
id_to_ifp = {}

for ifp in tqdm(ifps):
    id = ifp['id']
    id_to_ifp[id] = ifp
    fn = f'glimt/ifp/{id}.json'
    with open(fn, 'w') as f:
        json.dump(ifp, f, indent=4)

100%|███████████████████████████████████████████| 1/1 [00:00<00:00, 3566.59it/s]


## View open IFPs as a dataframe

In [255]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame([(ifp['id'], ifp['symbol'], ifp['dates']['startDay'], ifp['dates']['endDay'], 
                    ifp['props']['shortTitle']) for ifp in ifps], columns = ['id', 'challenge', 'startDay', 'endDay', 'title'])

df.sort_values(by=['challenge', 'endDay', 'title'])

,id,challenge,startDay,endDay,title
5,471,Eco_NORDSTREAM_Jan_25,20250114,20251230,Will Nord Stream be back in 2025?
8,475,Eco_RUSSIA_FUND_Jan_25,20250116,20250930,Russia to run out of cash by October?
2,468,Economics_RESERVES_Jan_25,20250114,20251230,When will Russia get back its frozen foreign exchange reserves?
0,458,Georgiainvasion,20250114,20251230,Will Russia invade Georgia in 2025?
18,502,Light_BelarusProtest,20250515,20251230,Protests in Belarus?
14,493,Light_ELONRICH_25,20250326,20251230,Will Elon Musk remain the richest in 2025?
17,501,Light_GDPG20Q2_May25,20250513,20250913,Economic growth in the G20-countries during Q2?
13,492,Light_TrumpNobel_25,20250326,20251009,Trump to win the 2025 Nobel Peace Prize?
15,498,Light_US_SouthKorea_Apr_25,20250428,20251230,American troops out of South Korea?
16,499,Pol_CHECHNYA_May_25,20250504,20251130,Ramzan Kadyrov still Head of Chechnya on Dec. 1?


## News for each question

In [120]:
from call_asknews import call_asknews

In [128]:
os.makedirs('glimt/news', exist_ok=True)

In [135]:
news = {}
for ifp in ifps:
    id, title, details = ifp['id'], ifp['props']['title'], ifp['props']['details']
    fn = f'glimt/news/{id}.txt'
    if os.path.exists(fn):
        with open(fn, 'r') as f:
            news[id] = f.read()
        continue
    prompt = f"""{title}\n{details}"""
    news[id] = call_asknews(prompt, True)
    with open(fn, 'w') as f:
        f.write(news[id])
    print('saved', fn)

saved glimt/news/463.txt
saved glimt/news/468.txt
saved glimt/news/469.txt
saved glimt/news/470.txt
saved glimt/news/471.txt
saved glimt/news/472.txt
saved glimt/news/473.txt
saved glimt/news/475.txt
saved glimt/news/476.txt
saved glimt/news/485.txt
saved glimt/news/489.txt
saved glimt/news/490.txt
saved glimt/news/492.txt
saved glimt/news/493.txt
saved glimt/news/498.txt
saved glimt/news/499.txt
saved glimt/news/501.txt
saved glimt/news/502.txt


## Prompts for each question

In [152]:
os.makedirs('glimt/prompt', exist_ok=True)

In [257]:
all_ifps = ifps.copy()

In [258]:
ifps = [ifps[0]]

In [259]:
ifp

{'id': 502,
 'type': 'bins',
 'symbol': 'Light_BelarusProtest',
 'state': 'active',
 'dates': {'startDay': 20250515, 'endDay': 20251230},
 'props': {'title': 'Will there be protests of political nature in Belarus during 2025, according to Carnegie Endowments Global Protest Tracker?',
  'ai_title': '',
  'shortTitle': 'Protests in Belarus?',
  'details': '<b>Background:</b> The ties between Russia and Belarus are strong and Belarus is an important partner for Russia. At the 2020 presidential election in Belarus, massive protests rocked the country. In 2022, a smaller number of individuals protested against the Russian war against Ukraine. Since then, there appears not to have been any protests in the country, not even in connection with the country’s presidential election earlier this year. \n\n<p><b>Resolution:</b> The question will be settled based on the report in the <a href ="https://carnegieendowment.org/features/global-protest-tracker?lang=en">Carnegie Endowments Global Protest T

In [267]:
prompt = {}

for ifp in ifps:
    id, startDay, endDay, title, details = [ifp['id'], ifp['dates']['startDay'],
              ifp['dates']['endDay'], ifp['props']['title'], ifp['props']['details']]
    fn = f'glimt/prompt/{id}.txt'
    bins = [x['props']['title'] for x in ifp['bins']]
    p1 = f"""
You are a talented, experienced and confident superforecaster. You are asked a question:

```question
{title}
```

You are given details on how to interpret the terms of the question:

```details
{details}
```

The following contemporary news is available for this question:
```news
{news[id]}
```

You must output a rationale R.  
This is a Markdown format text giving
* My rationale
* Reasons I might be right
* Reasons I might be wrong
Output this wrapped with tag
```rationale
```
"""
    if len(bins) == 2 and bins[0] == 'Yes':
        bp = f"""

This question resolves as Yes if the event happens and No otherwise.  You must output a numerical forecast which is the probability P, 0 <= P <= 1 of a yes answer wrapped in a tag in this format:
```probability_of_yes
P
```"""
    else:
        bp = f"""

This question can have one of {len(bins)} outcomes namely {', '.join(bins)}.  You must output a numerical forecast which a Python list of
{len(bins)} probabilities, where 0 <= Pi <= 1 and P1 + P2 + ... + P{len(bins)} = 1, in format
```binProbs
[P1,P2,...,P{len(bins)}]
```
"""
    bpw = "\nYou MUST ALWAYS OUTPUT A NUMERICAL FORECAST.  The task is failed if you omit this."
    prompt[id] = p1 + bp + bpw
    with open(fn, 'w') as f:
        f.write(prompt[id])

## Call my favorite LLM

In [266]:
from call_local_llm import call_local_llm

def humor_me(question):
    txt = call_local_llm(question, 'mistral-small3.2:24b-instruct-2506-q4_K_M')
    print()
    print(txt)
    return txt

## Get 5 rolls of the LLM for each prompt

In [268]:
rolls = {}
tries = 5
for ifp in tqdm(ifps):
    id = ifp['id']
    p = prompt[id]
    rolls[id] = [humor_me(p) for _ in range(tries)]

  0%|                                                     | 0/1 [00:00<?, ?it/s]

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.21276799043019612

```rationale
**Rationale:**

The question asks whether Russia will invade a NATO country from Belarus in 2025. The provided sources discuss the potential threat posed by Russia's military exercises in Belarus, particularly the "West-2025" exercises, and the possibility of Russia using Belarus as a staging ground for an invasion. Key points include:

1. **Threat from Belarus**: Multiple sources mention that Russia has been using Belarus as a potential threat to its neighbors, including NATO countries like Poland, Lithuania, Latvia, and Estonia. The "West-2025" exercises are seen as a cover for preparing for an invasion.
2. **NATO Vulnerabilities**: The Suwałki Corridor, which connects Poland and Lithuania, is highlighted as a vulnerable point that Russia could target to create a land corridor to Kaliningrad.
3. **Ukrainian and Western Concerns**: Ukrainian P

100%|█████████████████████████████████████████████| 1/1 [00:57<00:00, 57.51s/it]

model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.20643732945124307

```rationale
**Rationale:**

The question asks whether Russia will launch a military invasion from Belarus into a NATO country by the end of 2025. The provided sources suggest several key points:

1. **Threat from Belarus**: Multiple sources indicate that Russia is using military exercises in Belarus as a cover for potential aggressive actions. Ukrainian President Zelensky explicitly warns about Russia preparing "something" in Belarus for summer 2025, which could be an invasion.

2. **NATO Vulnerabilities**: The Suwałki Corridor, which connects Poland and Lithuania, is highlighted as a potential target. Russia could aim to control this corridor, which would threaten the Baltic states and Poland.

3. **Russian Capabilities**: While Russia faces significant challenges, including a shortage of equipment and inexperienced troops, the sources suggest that a concentrated effort (e.g., 80-100,000 troops) could pose a

## Break out 5 forecasts and rationales for each IFP

In [193]:
import numpy as np

In [269]:
def get_bin_probs(r):
    if 'binProbs' in r:
        return eval(r.split('```binProbs')[1].split('```')[0].strip())
    else:
        p_yes = float(r.split('```probability_of_yes')[1].split('```')[0].strip())
        p_no = 1 - p_yes
        return [p_yes, p_no]

In [270]:
forecasts = {}
rationales = {}

In [271]:
for id in rolls:
    R = rolls[id]
    forecast = [get_bin_probs(r) for r in R]
    forecasts[id] = forecast

In [272]:
def get_rationale(r):
    r1 = r.split('```rationale')[1].replace('Executive Summary of Rationale', 'Rationale').replace('Reasons You', 'Reasons I')
    r2 = r1.split('```')[0].replace('\nR\n', '\n')
    return r2.strip()

In [273]:
for id in rolls:
    R = rolls[id]
    rats = []
    for r in R:
        rat = get_rationale(r)
        rats.append(rat)
    rationales[id] = rats

In [274]:
forecasts

{458: [[0.35, 0.65], [0.35, 0.65], [0.65, 0.35], [0.65, 0.35], [0.35, 0.65]]}

## Median forecasts and rationales

In [292]:
def median_forecast(Fs):
    M = np.array(Fs)
    return np.median(M, axis=0).tolist()

In [299]:
def median_rationale(Rs):

    WRs = [f"""```forecast
{x}
```""" for x in Rs]
    
    WRS = '\n'.join(WRs)
    
    prompt = f"""
Summarize the gist of the rationale or thinking of the following answers from different forecasters to a single problem. 

{WRS}

DO NOT REFER TO THE FORECASTERS.  PRESENT THIS AS YOUR OWN THINKING, YOUR OWN RATIONALE.
"""
    
    medrat = humor_me(prompt)
    
    return medrat

## Combine and reject binProbs that are too long

In [302]:
median_forecasts = {}

for ifp in ifps:
    id = ifp['id']
    n_bins = len(ifp['bins'])
    Fs = forecasts[id]
    Rs = rationales[id]
    cum = []
    for fs, rs in zip(Fs,Rs):
        if len(fs) != n_bins:
            print('problem', id, n_bins, fs)
        else:
            cum.append((fs, rs))
    Fs = [fs for fs, rs in cum]
    Rs = [rs for fs, rs in cum]
    mfs = median_forecast(Fs)
    mrs = median_rationale(Rs)
    median_forecasts[id] = (mfs, mrs)

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.21366301774978638

The forecasts collectively highlight the potential for Russia to invade a NATO country from Belarus in 2025, primarily based on historical patterns, strategic vulnerabilities, and current geopolitical tensions. Here’s the gist of the rationale:

### Key Points Supporting the Possibility of an Invasion:
1. **Historical Precedent**: Russia has a track record of using military exercises as cover for invasions, such as the 2022 invasion of Ukraine. The "West-2025" exercises in Belarus are seen as a potential pretext for similar actions.
2. **Strategic Importance of Belarus**: Belarus serves as a staging ground for potential attacks on NATO countries, particularly Poland, Lithuania, Latvia, and Estonia. The Suwałki Corridor, a narrow strip of land connecting Poland and Lithuania, is identified as a critical vulnerability that Russia could exploit to create a lan

## Submit median forecast to IFP

To submit a forecast, replace jsxRequest by:

In [355]:
def submit_forecast(symbol, rationale, binProbas):
    jsx = f"""[["ifps","submitAIFcst",{{"ifpRef": "{symbol}","data": {{"probas": {binProbas} }},"reasoning": "{rationale}" }}]]"""
    print(jsx)
    return jsx_request(jsx)

where you should replace 
* **symbol** with the symbol of the IFP for which you are submitting a forecast bin
* **binProbas** with a JSON array of probabilities adding to 1.0, and such that each one corresponds to the IFP's bin (i.e. outcome) at the same index. For example, an IFP with 4 outcomes could accept [0.2, 0.6, 0, 0.2] where 0.6 is the probability you assign to the second outcome.
* **rationale** with your reasoning

It will return a JSON object containing the full description of your submitted forecast, including the forecast ID.

In [356]:
def submit_ifp(id):
    forecast, rationale = median_forecasts[id]
    symbol = id_to_ifp[id]['symbol']
    binProbas = forecast
    return submit_forecast(symbol, rationale, binProbas)

In [357]:
submit_ifp(458)

[["ifps","submitAIFcst",{"ifpRef": "Georgiainvasion","data": {"probas": [0.35, 0.65] },"reasoning": "The forecasts collectively highlight the potential for Russia to invade a NATO country from Belarus in 2025, primarily based on historical patterns, strategic vulnerabilities, and current geopolitical tensions. Here’s the gist of the rationale:

### Key Points Supporting the Possibility of an Invasion:
1. **Historical Precedent**: Russia has a track record of using military exercises as cover for invasions, such as the 2022 invasion of Ukraine. The "West-2025" exercises in Belarus are seen as a potential pretext for similar actions.
2. **Strategic Importance of Belarus**: Belarus serves as a staging ground for potential attacks on NATO countries, particularly Poland, Lithuania, Latvia, and Estonia. The Suwałki Corridor, a narrow strip of land connecting Poland and Lithuania, is identified as a critical vulnerability that Russia could exploit to create a land corridor to Kaliningrad.
3. 

{'SystemError': 'badRequestFormat'}